# 추가 봇 데이터 모델 B 프라임

원본 최종 수정 기록: 2025-06-06T01:27:25+09:00.

파일명의 번호는 파이프라인 순서이며 버전 번호가 아닙니다. 데이터·모델 경로와 실행 환경을 설정한 뒤 필요한 단계를 실행하세요.


In [ ]:
import pandas as pd

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
train_data_set = pd.read_csv('retrain_z_score.csv')
test_data_set = pd.read_csv('model_b_prime_test.csv')

In [ ]:
selected_features = [
    "description length",
    "#posts",
    "nums/length username",
    "#followers",
    "#follows",
    "fullname words",
    "private",
    "nums/length fullname",
    "external URL",
    # "name==username",
    # "profile pic"/

] #x_data_set에서 p-value를 검증을 했을 때 결과와 유의미하다고 판단된 특성들 만을 활용해서 학습

In [ ]:
df_X_selected = train_data_set[selected_features + ['fake']]
df_insta_selected = test_data_set[selected_features + ['fake']]

In [ ]:
X = df_X_selected.drop(columns=["fake"])
y = df_X_selected["fake"]

In [ ]:
X_train, X_vali, y_train, y_vali = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 개별 모델 정의
model1 = GradientBoostingClassifier(random_state=42)
model2 = RandomForestClassifier(random_state=42)
model3 = lgb.LGBMClassifier(random_state=42)
model4 = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')

In [ ]:
# 소프트 보팅 앙상블
voting_clf = VotingClassifier(
    estimators=[
        ('gb', model1),
        ('rf', model2),
        ('lgbm', model3),
        ('xgb', model4)
    ],
    voting='soft'
)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# 학습
voting_clf.fit(X_train, y_train)

# 확률 예측
probs = voting_clf.predict_proba(X_vali)

# 클래스 1 (Fake, 봇) 확률만 추출
bot_probs = probs[:, 1]

# ✅ 원하는 threshold 지정 (예: 0.6)
threshold = 0.6
preds_thresh = (bot_probs >= threshold).astype(int)

# 평가
print(f"📌 Voting Classifier Accuracy (threshold={threshold}):", accuracy_score(y_vali, preds_thresh))
print(classification_report(y_vali, preds_thresh, target_names=["Real", "Fake"]))


In [ ]:
X_test = df_insta_selected.drop(columns=["fake"])
y_test = df_insta_selected["fake"]

In [ ]:
preds = voting_clf.predict(X_test)

# 결과 출력
print("Voting Classifier Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=["Real", "Fake"]))